# 01 — Tox21 Exploratory Data Analysis

탐색적 데이터 분석: 멀티레이블 분포, 결측치, 물리화학적 특성(LogP/MW), 대표 분자 구조.

Exploratory analysis of the Tox21 multi-label toxicity dataset.

In [ ]:
# === Colab setup (run once, restart runtime if prompted) ===
import os, sys
if not os.path.exists("tox21-toxicity-classifier"):
    !git clone https://github.com/zqvo04/tox21-toxicity-classifier.git
%cd tox21-toxicity-classifier
!bash setup_colab.sh
# Make src importable
sys.path.insert(0, os.getcwd())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

from src.dataset import label_matrix
from src import TOX21_TASKS

FIG = 'results/figures'
os.makedirs(FIG, exist_ok=True)

## 1. 데이터 로딩 & 레이블 행렬

In [ ]:
y, tasks, smiles = label_matrix()
print('Molecules:', len(smiles), '| Tasks:', len(tasks))
df_y = pd.DataFrame(y, columns=tasks)
df_y.head()

## 2. 레이블 분포 & 결측치 히트맵

In [ ]:
# Positive rate & missing rate per task
pos_rate = df_y.apply(lambda c: np.nanmean(c))
missing_rate = df_y.isna().mean()
stats = pd.DataFrame({'positive_rate': pos_rate, 'missing_rate': missing_rate})
display(stats)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
stats['positive_rate'].plot.bar(ax=axes[0], color='crimson')
axes[0].set_title('Positive (toxic) rate per task'); axes[0].set_ylabel('rate')
stats['missing_rate'].plot.bar(ax=axes[1], color='steelblue')
axes[1].set_title('Missing-label rate per task'); axes[1].set_ylabel('rate')
plt.tight_layout(); plt.savefig(f'{FIG}/label_stats.png', dpi=150); plt.show()

In [ ]:
# Co-occurrence / correlation heatmap of toxicity endpoints
corr = df_y.corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, cbar_kws={'shrink': .8})
plt.title('Tox21 endpoint label correlation')
plt.tight_layout(); plt.savefig(f'{FIG}/label_correlation_heatmap.png', dpi=150); plt.show()

## 3. 물리화학적 특성 분포 (LogP / MW)

In [ ]:
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw

records = []
for smi in smiles:
    m = Chem.MolFromSmiles(smi)
    if m is None:
        continue
    records.append({'smiles': smi,
                    'MW': Descriptors.MolWt(m),
                    'LogP': Descriptors.MolLogP(m),
                    'TPSA': Descriptors.TPSA(m),
                    'HBD': Descriptors.NumHDonors(m),
                    'HBA': Descriptors.NumHAcceptors(m)})
props = pd.DataFrame(records)
props.describe()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(props['MW'], bins=50, kde=True, ax=axes[0], color='teal')
axes[0].set_title('Molecular Weight'); axes[0].axvline(500, ls='--', c='r')
sns.histplot(props['LogP'], bins=50, kde=True, ax=axes[1], color='darkorange')
axes[1].set_title('LogP'); axes[1].axvline(5, ls='--', c='r')
sns.histplot(props['TPSA'], bins=50, kde=True, ax=axes[2], color='purple')
axes[2].set_title('TPSA')
plt.tight_layout(); plt.savefig(f'{FIG}/physchem_distributions.png', dpi=150); plt.show()

## 4. 대표 분자 구조 시각화
독성(positive) 분자 샘플의 2D 구조.

In [ ]:
# Show some molecules flagged toxic on SR-MMP (mitochondrial membrane potential)
task = 'SR-MMP'
ti = tasks.index(task)
toxic_smiles = [s for s, lab in zip(smiles, y[:, ti]) if lab == 1][:12]
mols = [Chem.MolFromSmiles(s) for s in toxic_smiles]
mols = [m for m in mols if m is not None]
img = Draw.MolsToGridImage(mols, molsPerRow=4, subImgSize=(220, 180),
                           legends=[f'{task}=1']*len(mols))
img.save(f'{FIG}/example_toxic_molecules.png')
img

## 요약
- 12개 독성 엔드포인트, 클래스 불균형이 큼 (positive rate 낮음)
- 결측 레이블 비율이 task마다 다름 → **마스킹 필수**
- 대부분 분자가 Lipinski 규칙(MW<500, LogP<5) 범위에 위치